# 1. Data Parallelism

This notebook introduces the simplest training parallelism first.

**Core idea**
- every GPU keeps the same model
- the batch is split into smaller pieces
- each GPU computes gradients on its own data shard
- gradients are averaged before the optimizer step

In a real multi-GPU run, each model copy lives on a different GPU.
In this notebook, we simulate the same math on one runtime so the
idea is easy to see in Colab.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0
CUDA available: False


## Why start here?

Data parallelism is usually the easiest parallelism to understand
because the model itself does not change. Only the **batch**
changes.

We will compare:
1. one normal full-batch training step
2. one virtual data-parallel training step with two replicas

If both are implemented correctly, the parameter updates should be
almost identical.

In [2]:
from copy import deepcopy


class TinyClassifier(nn.Module):
    """Small network used to demonstrate gradient averaging."""

    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


batch_size = 8
input_dim = 4
hidden_dim = 10
num_classes = 3
learning_rate = 0.1

x = torch.randn(batch_size, input_dim)
y = torch.tensor([0, 1, 2, 1, 0, 2, 1, 0], dtype=torch.long)

base_model = TinyClassifier(input_dim, hidden_dim, num_classes)
base_state = deepcopy(base_model.state_dict())
criterion = nn.CrossEntropyLoss(reduction="sum")

print("Input batch shape:", tuple(x.shape))
print("Labels shape:", tuple(y.shape))

Input batch shape: (8, 4)
Labels shape: (8,)


In [3]:
def run_full_batch_step(initial_state: dict[str, torch.Tensor]) -> tuple[float, dict[str, torch.Tensor]]:
    model = TinyClassifier(input_dim, hidden_dim, num_classes)
    model.load_state_dict(initial_state)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    optimizer.zero_grad()
    logits = model(x)
    loss = criterion(logits, y) / batch_size
    loss.backward()
    optimizer.step()

    return loss.item(), deepcopy(model.state_dict())


def run_virtual_data_parallel_step(
    initial_state: dict[str, torch.Tensor],
    num_replicas: int,
) -> tuple[float, dict[str, torch.Tensor]]:
    reference_model = TinyClassifier(input_dim, hidden_dim, num_classes)
    reference_model.load_state_dict(initial_state)
    optimizer = torch.optim.SGD(reference_model.parameters(), lr=learning_rate)

    for parameter in reference_model.parameters():
        parameter.grad = torch.zeros_like(parameter)

    total_loss = 0.0
    x_shards = x.chunk(num_replicas)
    y_shards = y.chunk(num_replicas)

    for replica_id, (x_shard, y_shard) in enumerate(zip(x_shards, y_shards)):
        replica = TinyClassifier(input_dim, hidden_dim, num_classes)
        replica.load_state_dict(initial_state)

        shard_logits = replica(x_shard)
        shard_loss_sum = criterion(shard_logits, y_shard)
        shard_loss_sum.backward()
        total_loss += shard_loss_sum.item()

        print(
            f"Replica {replica_id}: shard shape={tuple(x_shard.shape)}, "
            f"examples={len(x_shard)}"
        )

        for reference_parameter, replica_parameter in zip(
            reference_model.parameters(),
            replica.parameters(),
        ):
            reference_parameter.grad += replica_parameter.grad

    for parameter in reference_model.parameters():
        parameter.grad /= batch_size

    optimizer.step()
    mean_loss = total_loss / batch_size
    return mean_loss, deepcopy(reference_model.state_dict())

In [ ]:
full_loss, full_state = run_full_batch_step(base_state)
dp_loss, dp_state = run_virtual_data_parallel_step(base_state, num_replicas=2)


def max_state_difference(
    left: dict[str, torch.Tensor],
    right: dict[str, torch.Tensor],
) -> float:
    differences = [
        (left[name] - right[name]).abs().max().item()
        for name in left
    ]
    return max(differences)


print()
print(f"Full-batch loss:         {full_loss:.6f}")
print(f"Virtual data-parallel:   {dp_loss:.6f}")
print(f"Max parameter difference: {max_state_difference(full_state, dp_state):.10f}")

assert torch.isclose(torch.tensor(full_loss), torch.tensor(dp_loss), atol=1e-7)
assert max_state_difference(full_state, dp_state) < 1e-7

print("The data-parallel update matches the full-batch update.")

## Takeaways

- Data parallelism keeps the model architecture unchanged.
- The main communication step is **gradient averaging**.
- This is often the first parallelism to try when one GPU is too slow
  but the model still fits in memory.